This file contains the methods we use to clean this data. The data is actually reasonably clean out of the package but there are a few things that need to be ironed out.

In [5]:
import numpy as np
import pandas as pd
import json
from pathlib import Path

In [8]:
# The files pull from the whole year, we only want ski season (generally Nov-Apr)
# Add a season column
def get_seasons(df):
    season_months = ["11", "12", "01", "02", "03", "04"]
    month = df["time"].str.split("-").str[1]
    df = df[month.isin(season_months)].copy()
    month = df["time"].str.split("-").str[1]
    df["season"] = np.where(month.isin(["11", "12", "01", "02"]), "winter", "spring")
    return df


# We don't need sunrise and sunset because we have daylight_duration
# Create daylight_hours column that gives us time in hours instead of seconds
def fix_times(df):
    df = df.drop(columns=["sunrise", "sunset"])
    df["daylight_hours"] = df["daylight_duration"] / 3600
    return df


# The snowfall_sum is exactly 0.7 x snowfall_water_equivalent so we don't need both columns. I will just keep the snowfall_sum
# There are other columns that are highly correlated with each other. I will keep one from each similar group
def remove_duplicate_features(df):
    # Only one dropped for now, but the method makes it easy to remove more columns later
    return df.drop(columns=["snowfall_water_equivalent_sum"])


# Some observations record rainfall even though the max temp is below freezing, which doesn't make sense.
# In these rows I set rainfall to 0 and convert total precipitation to snowfall
def fix_improbable(df):
    frozen = (df["temperature_2m_max"] < 0) & (df["rain_sum"] > 0)

    df.loc[frozen, "snowfall_water_equivalent_sum"] = df.loc[frozen, "precipitation_sum"]
    df.loc[frozen, "snowfall_sum"] = df.loc[frozen, "precipitation_sum"] * 0.7
    df.loc[frozen, "rain_sum"] = 0

    to_snow = {51: 71, 53: 73, 55: 75, 61: 71, 63: 73, 65: 75}
    mask = (df["temperature_2m_max"] < 0) & df["weather_code"].isin(to_snow.keys())
    df.loc[mask, "weather_code"] = df.loc[mask, "weather_code"].map(to_snow)
    return df

The above are helpers to help handle specific cleaning tasks. The below method loops through the raw data, cleans each file, and caches the clean version

In [9]:
# Read an Open-Meteo JSON file into a DataFrame, keeping location info as columns.
def load_raw(path):
    with open(path) as f:
        raw = json.load(f)
    df = pd.DataFrame(raw["daily"])
    df["latitude"] = raw["latitude"]
    df["longitude"] = raw["longitude"]
    df["elevation"] = raw["elevation"]
    df["source_file"] = Path(path).name
    return df


def clean_df(df):
    # Order matters: fix_improbable writes snowfall_water_equivalent_sum,
    # so it has to run before remove_duplicate_features drops that column.
    df = fix_improbable(df)
    df = remove_duplicate_features(df)
    df = fix_times(df)
    df = get_seasons(df)
    return df.reset_index(drop=True)


"""Clean every JSON file in raw_dir and save it to clean_dir as <name>_clean.json.

Files that already have a clean version are skipped. If clean_dir isn't given,
it's raw_dir with the "raw" folder swapped for "clean".
"""
def clean_directory(raw_dir="data/raw/openmeteo/daily_best_match", clean_dir=None):

    raw_dir = Path(raw_dir)
    if clean_dir is None:
        clean_dir = Path(*["clean" if p == "raw" else p for p in raw_dir.parts])
    clean_dir = Path(clean_dir)
    clean_dir.mkdir(parents=True, exist_ok=True)

    for path in sorted(raw_dir.glob("*.json")):
        out_path = clean_dir / f"{path.stem}_clean.json"
        if out_path.exists():
            print(f"skip   {path.name} (already cleaned)")
            continue
        df = clean_df(load_raw(path))
        df.to_json(out_path, orient="records", indent=2)
        print(f"clean  {path.name} -> {out_path} ({len(df)} rows)")

In [10]:
clean_directory()

clean  grandvalira_base_2016-08-01_2016-12-31.json -> data/clean/openmeteo/daily_best_match/grandvalira_base_2016-08-01_2016-12-31_clean.json (61 rows)
clean  grandvalira_base_2017-01-01_2017-12-31.json -> data/clean/openmeteo/daily_best_match/grandvalira_base_2017-01-01_2017-12-31_clean.json (181 rows)
clean  grandvalira_base_2018-01-01_2018-12-31.json -> data/clean/openmeteo/daily_best_match/grandvalira_base_2018-01-01_2018-12-31_clean.json (181 rows)
clean  grandvalira_base_2019-01-01_2019-12-31.json -> data/clean/openmeteo/daily_best_match/grandvalira_base_2019-01-01_2019-12-31_clean.json (181 rows)
clean  grandvalira_base_2020-01-01_2020-12-31.json -> data/clean/openmeteo/daily_best_match/grandvalira_base_2020-01-01_2020-12-31_clean.json (182 rows)
clean  grandvalira_mid_2016-08-01_2016-12-31.json -> data/clean/openmeteo/daily_best_match/grandvalira_mid_2016-08-01_2016-12-31_clean.json (61 rows)
clean  grandvalira_mid_2017-01-01_2017-12-31.json -> data/clean/openmeteo/daily_best_m